In [14]:
# cell 1
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
import pydicom
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, \
    roc_curve
from tqdm import tqdm
import os
import warnings
import time

warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU found - training will be slow!")

print("✓ All libraries imported")

Device: cpu
⚠️ No GPU found - training will be slow!
✓ All libraries imported


In [15]:
# Define your dataset paths
BASE_DIR = 'D:/rsna'

TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'stage_2_train_images')
TEST_IMG_DIR = os.path.join(BASE_DIR, 'stage_2_test_images')

TRAIN_LABELS_CSV = os.path.join(BASE_DIR, 'stage_2_train_labels.csv')
DETAILED_CLASS_CSV = os.path.join(BASE_DIR, 'stage_2_detailed_class_info.csv')

# Verify paths exist
print("Checking paths...")
print(f"Train images exist: {os.path.exists(TRAIN_IMG_DIR)}")
print(f"Test images exist: {os.path.exists(TEST_IMG_DIR)}")
print(f"Train labels exist: {os.path.exists(TRAIN_LABELS_CSV)}")
print(f"Detailed class info exist: {os.path.exists(DETAILED_CLASS_CSV)}")

# Count images
train_image_count = len([f for f in os.listdir(TRAIN_IMG_DIR) if f.endswith('.dcm')])
test_image_count = len([f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.dcm')])

print(f"\nTotal train images: {train_image_count}")
print(f"Total test images: {test_image_count}")
print("✓ Paths verified")

Checking paths...
Train images exist: True
Test images exist: True
Train labels exist: True
Detailed class info exist: True

Total train images: 26684
Total test images: 3000
✓ Paths verified


In [16]:
# Load train labels
train_labels = pd.read_csv(TRAIN_LABELS_CSV)

print("Original Train Labels CSV:")
print(train_labels.head(10))
print(f"\nShape: {train_labels.shape}")
print(f"Columns: {train_labels.columns.tolist()}")

# Group by patientId - if ANY row has Target=1, patient has pneumonia
train_patient_labels = train_labels.groupby('patientId').agg({
    'Target': 'max'  # 1 if pneumonia exists, 0 if normal
}).reset_index()

print("\n" + "=" * 60)
print("PATIENT-LEVEL TRAIN LABELS:")
print("=" * 60)
print(f"Unique patients: {len(train_patient_labels)}")
print(f"Pneumonia cases: {(train_patient_labels['Target'] == 1).sum()}")
print(f"Normal cases: {(train_patient_labels['Target'] == 0).sum()}")
print("\nSample:")
print(train_patient_labels.head(10))

Original Train Labels CSV:
                              patientId      x      y  width  height  Target
0  0004cfab-14fd-4e49-80ba-63a80b6bddd6    NaN    NaN    NaN     NaN       0
1  00313ee0-9eaa-42f4-b0ab-c148ed3241cd    NaN    NaN    NaN     NaN       0
2  00322d4d-1c29-4943-afc9-b6754be640eb    NaN    NaN    NaN     NaN       0
3  003d8fa0-6bf1-40ed-b54c-ac657f8495c5    NaN    NaN    NaN     NaN       0
4  00436515-870c-4b36-a041-de91049b9ab4  264.0  152.0  213.0   379.0       1
5  00436515-870c-4b36-a041-de91049b9ab4  562.0  152.0  256.0   453.0       1
6  00569f44-917d-4c86-a842-81832af98c30    NaN    NaN    NaN     NaN       0
7  006cec2e-6ce2-4549-bffa-eadfcd1e9970    NaN    NaN    NaN     NaN       0
8  00704310-78a8-4b38-8475-49f4573b2dbb  323.0  577.0  160.0   104.0       1
9  00704310-78a8-4b38-8475-49f4573b2dbb  695.0  575.0  162.0   137.0       1

Shape: (30227, 6)
Columns: ['patientId', 'x', 'y', 'width', 'height', 'Target']

PATIENT-LEVEL TRAIN LABELS:
Unique patients:

In [17]:
# cell 4
# Load detailed class info (contains test labels)
detailed_class = pd.read_csv(DETAILED_CLASS_CSV)

print("Detailed Class Info CSV:")
print(detailed_class.head(10))
print(f"\nShape: {detailed_class.shape}")
print(f"Columns: {detailed_class.columns.tolist()}")
print(f"\nUnique classes: {detailed_class['class'].unique()}")

# Create test labels - Map class to binary target
# 'Normal' = 0, 'No Lung Opacity / Not Normal', 'Lung Opacity' = 1
test_patient_labels = detailed_class.copy()

# Create Target column based on class
test_patient_labels['Target'] = test_patient_labels['class'].apply(
    lambda x: 0 if x == 'Normal' else 1
)

print("\n" + "=" * 60)
print("TEST PATIENT LABELS:")
print("=" * 60)
print(f"Total patients: {len(test_patient_labels)}")
print(f"Pneumonia cases: {(test_patient_labels['Target'] == 1).sum()}")
print(f"Normal cases: {(test_patient_labels['Target'] == 0).sum()}")
print("\nClass distribution:")
print(test_patient_labels['class'].value_counts())

Detailed Class Info CSV:
                              patientId                         class
0  0004cfab-14fd-4e49-80ba-63a80b6bddd6  No Lung Opacity / Not Normal
1  00313ee0-9eaa-42f4-b0ab-c148ed3241cd  No Lung Opacity / Not Normal
2  00322d4d-1c29-4943-afc9-b6754be640eb  No Lung Opacity / Not Normal
3  003d8fa0-6bf1-40ed-b54c-ac657f8495c5                        Normal
4  00436515-870c-4b36-a041-de91049b9ab4                  Lung Opacity
5  00436515-870c-4b36-a041-de91049b9ab4                  Lung Opacity
6  00569f44-917d-4c86-a842-81832af98c30  No Lung Opacity / Not Normal
7  006cec2e-6ce2-4549-bffa-eadfcd1e9970  No Lung Opacity / Not Normal
8  00704310-78a8-4b38-8475-49f4573b2dbb                  Lung Opacity
9  00704310-78a8-4b38-8475-49f4573b2dbb                  Lung Opacity

Shape: (30227, 2)
Columns: ['patientId', 'class']

Unique classes: ['No Lung Opacity / Not Normal' 'Normal' 'Lung Opacity']

TEST PATIENT LABELS:
Total patients: 30227
Pneumonia cases: 21376
Normal cases:

In [18]:
# cell 5
from sklearn.model_selection import train_test_split

# ============================================================
# REDUCE TRAINING SET TO 5000 SAMPLES (for faster training)
# ============================================================
TRAIN_SIZE = 5000  # Use only 5000 samples for training

print("=" * 60)
print("REDUCING DATASET FOR FASTER TRAINING")
print("=" * 60)
print(f"Original training size: {len(train_patient_labels)}")

# Sample 5000 while maintaining class balance
train_patient_labels_reduced = train_patient_labels.groupby('Target', group_keys=False).apply(
    lambda x: x.sample(min(len(x), TRAIN_SIZE // 2), random_state=42)
).reset_index(drop=True)

print(f"Reduced training size: {len(train_patient_labels_reduced)}")
print(f"  Pneumonia: {(train_patient_labels_reduced['Target'] == 1).sum()}")
print(f"  Normal: {(train_patient_labels_reduced['Target'] == 0).sum()}")

# Split reduced train into train (80%) and validation (20%)
train_df, val_df = train_test_split(
    train_patient_labels_reduced,
    test_size=0.2,
    random_state=42,
    stratify=train_patient_labels_reduced['Target']
)

print("\n" + "=" * 60)
print("TRAIN/VAL SPLIT:")
print("=" * 60)
print(f"TRAINING: {len(train_df)} patients")
print(f"  Pneumonia: {(train_df['Target'] == 1).sum()}")
print(f"  Normal: {(train_df['Target'] == 0).sum()}")

print(f"\nVALIDATION: {len(val_df)} patients")
print(f"  Pneumonia: {(val_df['Target'] == 1).sum()}")
print(f"  Normal: {(val_df['Target'] == 0).sum()}")

# Test set (use full test set)
test_df = test_patient_labels[['patientId', 'Target']]

print(f"\nTEST: {len(test_df)} patients")
print(f"  Pneumonia: {(test_df['Target'] == 1).sum()}")
print(f"  Normal: {(test_df['Target'] == 0).sum()}")

# Estimate training time
batches_per_epoch = len(train_df) // 32 + 1
est_time_per_epoch = batches_per_epoch * 1.0 / 60  # ~1 second per batch
print("\n" + "=" * 60)
print(f"Estimated time per epoch: ~{est_time_per_epoch:.1f} minutes")
print(f"Estimated total (10 epochs): ~{est_time_per_epoch * 10:.1f} minutes")
print("=" * 60)

REDUCING DATASET FOR FASTER TRAINING
Original training size: 26684
Reduced training size: 5000
  Pneumonia: 2500
  Normal: 2500

TRAIN/VAL SPLIT:
TRAINING: 4000 patients
  Pneumonia: 2000
  Normal: 2000

VALIDATION: 1000 patients
  Pneumonia: 500
  Normal: 500

TEST: 30227 patients
  Pneumonia: 21376
  Normal: 8851

Estimated time per epoch: ~2.1 minutes
Estimated total (10 epochs): ~21.0 minutes


In [19]:
# cell 6
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("✓ Transforms ready")

✓ Transforms ready


In [20]:
# cell 7
class RSNADataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        patient_id = self.df.iloc[idx]['patientId']
        label = int(self.df.iloc[idx]['Target'])
        img_path = os.path.join(self.img_dir, f'{patient_id}.dcm')

        try:
            # Read DICOM file
            dcm = pydicom.dcmread(img_path)
            img = dcm.pixel_array

            # Normalize to 0-255
            img = (img - img.min()) / (img.max() - img.min() + 1e-6) * 255.0
            img = img.astype(np.uint8)

            # Convert grayscale to RGB (3 channels)
            img = np.stack([img, img, img], axis=-1)
            img = Image.fromarray(img)

            # Apply transforms
            if self.transform:
                img = self.transform(img)

            return img, label

        except Exception as e:
            print(f"Error loading {patient_id}: {e}")
            # Return zero tensor on error
            return torch.zeros(3, 224, 224), label


print("✓ RSNADataset class defined")

✓ RSNADataset class defined


In [21]:
#cell 8
# Create datasets
train_dataset = RSNADataset(train_df, TRAIN_IMG_DIR, train_transform)
val_dataset = RSNADataset(val_df, TRAIN_IMG_DIR, val_test_transform)
test_dataset = RSNADataset(test_df, TEST_IMG_DIR, val_test_transform)

BATCH_SIZE = 32

# Create dataloaders with MINIMAL settings
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("✓ DataLoaders created")
print(f"TRAIN: {len(train_loader)} batches")
print(f"VAL: {len(val_loader)} batches")
print(f"TEST: {len(test_loader)} batches")

✓ DataLoaders created
TRAIN: 125 batches
VAL: 32 batches
TEST: 945 batches


In [22]:
#cell 9
def create_model(model_name='resnet50', pretrained=True, num_classes=2):
    """Create a CNN model for binary pneumonia classification"""

    if model_name == 'resnet18':
        model = models.resnet18(pretrained=pretrained)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)

    elif model_name == 'resnet50':
        model = models.resnet50(pretrained=pretrained)
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)

    elif model_name == 'densenet121':
        model = models.densenet121(pretrained=pretrained)
        num_features = model.classifier.in_features
        model.classifier = nn.Linear(num_features, num_classes)

    return model


# Create model
MODEL_NAME = 'resnet50'  # You can try 'resnet18' for faster training
model = create_model(MODEL_NAME, pretrained=True, num_classes=2)
model = model.to(device)

print(f"✓ Model created: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

✓ Model created: resnet50
Total parameters: 23,512,130
Trainable parameters: 23,512,130


In [23]:
#cell 10
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler (reduces LR when validation loss plateaus)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,

)

print("✓ Loss function: CrossEntropyLoss")
print("✓ Optimizer: Adam (lr=0.001)")
print("✓ Scheduler: ReduceLROnPlateau")

✓ Loss function: CrossEntropyLoss
✓ Optimizer: Adam (lr=0.001)
✓ Scheduler: ReduceLROnPlateau


In [24]:
#CELL 11
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Train the model for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc='Training', leave=False)

    for images, labels in progress_bar:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100 * correct / total:.2f}%'
        })

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc


print("✓ Training function defined")

✓ Training function defined


In [25]:
# cell 12
def evaluate(model, data_loader, criterion, device, desc='Evaluation'):
    """Evaluate the model on a dataset"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_predictions = []
    all_probabilities = []

    progress_bar = tqdm(data_loader, desc=desc, leave=False)

    with torch.no_grad():
        for images, labels in progress_bar:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Get predictions
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)

            # Statistics
            running_loss += loss.item()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Store for metrics
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
            all_probabilities.extend(probabilities[:, 1].cpu().numpy())

            # Update progress bar
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct / total:.2f}%'
            })

    epoch_loss = running_loss / len(data_loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc, all_labels, all_predictions, all_probabilities


print("✓ Evaluation function defined")

✓ Evaluation function defined


In [26]:
# cell 13
NUM_EPOCHS = 10

# Store training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_model_path = 'best_pneumonia_model.pth'

print("=" * 60)
print("🚀 STARTING TRAINING")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {device}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print("=" * 60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\n{'=' * 60}")
    print(f"EPOCH {epoch + 1}/{NUM_EPOCHS}")
    print('=' * 60)

    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validate
    val_loss, val_acc, _, _, _ = evaluate(
        model, val_loader, criterion, device, desc='Validation'
    )

    # Update learning rate
    scheduler.step(val_loss)

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Print epoch summary
    print(f"\n📊 Epoch {epoch + 1} Summary:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"   ✅ New best model saved! (Val Acc: {val_acc:.2f}%)")

total_time = time.time() - start_time

print("\n" + "=" * 60)
print("🎉 TRAINING COMPLETE!")
print("=" * 60)
print(f"Total training time: {total_time / 60:.2f} minutes ({total_time / 3600:.2f} hours)")
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"Best model saved to: {best_model_path}")
print("=" * 60)

🚀 STARTING TRAINING
Model: resnet50
Epochs: 10
Batch size: 32
Device: cpu
Training samples: 4000
Validation samples: 1000

EPOCH 1/10


KeyboardInterrupt: 

stupid ass nigger
